## Exploratory Data Analysis

In [3]:
import numpy as np
import helpers
import seaborn as sb
import csv

### 1. Loading Data

In [4]:
x_train, x_test, y_train, train_ids, test_ids = helpers.load_csv_data("data/dataset")
full_headers = helpers.read_header("data/dataset/x_train.csv")
print(x_train[0:10, :])
print(full_headers)

[[5.3000000e+01 1.1000000e+01 1.1162015e+07 ...           nan
            nan 2.0000000e+00]
 [3.3000000e+01 1.2000000e+01 1.2152015e+07 ...           nan
            nan           nan]
 [2.0000000e+01 1.0000000e+01 1.0202015e+07 ... 1.0000000e+00
  2.0000000e+00 2.0000000e+00]
 ...
 [4.4000000e+01 3.0000000e+00 3.0720150e+06 ... 1.0000000e+00
  2.0000000e+00 2.0000000e+00]
 [2.1000000e+01 7.0000000e+00 7.3120150e+06 ...           nan
            nan 2.0000000e+00]
 [4.0000000e+00 6.0000000e+00 6.2320150e+06 ...           nan
            nan           nan]]
['_STATE' 'FMONTH' 'IDATE' 'IMONTH' 'IDAY' 'IYEAR' 'DISPCODE' 'SEQNO'
 '_PSU' 'CTELENUM' 'PVTRESD1' 'COLGHOUS' 'STATERES' 'CELLFON3' 'LADULT'
 'NUMADULT' 'NUMMEN' 'NUMWOMEN' 'CTELNUM1' 'CELLFON2' 'CADULT' 'PVTRESD2'
 'CCLGHOUS' 'CSTATE' 'LANDLINE' 'HHADULT' 'GENHLTH' 'PHYSHLTH' 'MENTHLTH'
 'POORHLTH' 'HLTHPLN1' 'PERSDOC2' 'MEDCOST' 'CHECKUP1' 'BPHIGH4' 'BPMEDS'
 'BLOODCHO' 'CHOLCHK' 'TOLDHI2' 'CVDSTRK3' 'ASTHMA3' 'ASTHNOW' 'CHCSCNCR

In [5]:
print("Size of Data:", np.shape(x_train))
print(x_train[0:10,0])

Size of Data: (328135, 321)
[53. 33. 20. 42. 24. 54. 39. 44. 21.  4.]


The Id column and variables names aren't loaded by the helper function `load_csv_data` so we need to manually keep track of the column headers.

### 2. Select only Calculated Variables and Clean

In [6]:
x_train_calculated = x_train[:,-91:] 
calculated_headers = full_headers[-91:]
print(len(calculated_headers))

91


The following steps of initial cleaning are taken:

1. Remove variables are just used to measure the quality of the response. They are not needed for this stage but can be added back later. This includes the following:

  - `_MISVEGN`, `_MISFRTN`, `_FRTRESP`, `_VEGRESP`, `_FRUITEX`, `_VEGETEX`, `_FRT16`, `_VEG23`, `PAMISS1_`

2. If multiple variables actually measure the same thing, then only one is needed to be kept. This includes the following:

  - `HTIN4` and `HTM4`: keep only `HTM4`
  - Most of the variables containing age and race information: keep only `_AGE80` and `_RACE`

3. Remove variables that are an exact linear combination or a function of some other variables. This includes the following:

  - `FC60_` and `MAXVO2_`: one is (almost) an exact multiple of the other, we keep `FC60_`.
  - `PA1MIN_` and `PA1VIGM_`: exact sums of the per-activity minutes (`PAMIN11_ + PAMIN21_` and `PAVIG11_ + PAVIG21_`)
  - `FTJUDA1_`, `FRUTDA1_`, `BEANDAY_`, `GRENDAY_`, `ORNGDAY_`, `VEGEDA1_`: exact components of `_FRUTSUM` and `_VEGESUM` — we keep the two sums
  - `_RFSMOK3`: `== 2` exactly when `_SMOKER3 in {1, 2}` (current smoker)
  - `_RFBMI5` and `_BMI5CAT`: threshold (`_BMI5 >= 25`) and buckets (18.5 / 25 / 30) of `_BMI5`
  - `_FRTLT1` and `_VEGLT1`: `== 1` exactly when `_FRUTSUM >= 1` / `_VEGESUM >= 1`
  - `DRNKANY5`: `== 1` exactly when `_DRNKWEK > 0`
  - `_PASTRNG`: `== 1` exactly when `STRFREQ_ >= 2` (strength training twice a week)
  - `_PAREC1`: exact 2x2 cross of `_PAINDX1` and `_PASTRNG`; `_PAINDX1`, `_PA30021` and `_PASTAE1` are exact collapses of `_PA150R2`, `_PA300R2` and `_PAREC1`
  - `_DRDXAR1`: `== 1` exactly when `_LMTACT1 in {1, 2}` (diagnosed with arthritis)
  - `_LTASTH1` and `_CASTHM1`: `_ASTHMS1` is their exact three-way merge (never/former/current asthma)

In [7]:
drop_list = [
       # response quality
       "_MISVEGN", "_MISFRTN", "_FRTRESP", "_VEGRESP",
       "_FRUITEX", "_VEGETEX", "_FRT16", "_VEG23", "PAMISS1_",
       # age/race duplicates (keep _AGE80, _RACE)
       "_AGEG5YR", "_AGE_G", "_AGE65YR",
       "_PRACE1", "_MRACE1", "_HISPANC", "_RACEG21", "_RACEGR3", "_RACE_G1",
       # exact duplicates/ rescales
       "HTIN4",                                          
       "MAXVO2_",                                        
       # overlapping info
       "PA1MIN_", "PA1VIGM_",
       "FTJUDA1_", "FRUTDA1_",                            
       "BEANDAY_", "GRENDAY_", "ORNGDAY_", "VEGEDA1_",    
       "_RFSMOK3", "_RFBMI5", "_BMI5CAT",
       "_FRTLT1", "_VEGLT1", "DRNKANY5", "_PASTRNG",
       "_PAREC1", "_PAINDX1", "_PA30021", "_PASTAE1",
       "_DRDXAR1", "_LTASTH1", "_CASTHM1",
]

col_idx = np.where(np.isin(calculated_headers, drop_list))[0]
x_train_cal_cleaned = np.delete(x_train_calculated, col_idx, axis=1)
cal_headers_cleaned = np.delete(calculated_headers, col_idx)   
print("Remaining columns:", len(cal_headers_cleaned))
print(cal_headers_cleaned)

Remaining columns: 49
['_RFHLTH' '_HCVU651' '_RFHYPE5' '_CHOLCHK' '_RFCHOL' '_ASTHMS1' '_RACE'
 '_AGE80' 'HTM4' 'WTKG3' '_BMI5' '_CHLDCNT' '_EDUCAG' '_INCOMG' '_SMOKER3'
 'DROCDY3_' '_RFBING5' '_DRNKWEK' '_RFDRHV5' '_FRUTSUM' '_VEGESUM'
 '_TOTINDA' 'METVL11_' 'METVL21_' 'FC60_' 'ACTIN11_' 'ACTIN21_' 'PADUR1_'
 'PADUR2_' 'PAFREQ1_' 'PAFREQ2_' '_MINAC11' '_MINAC21' 'STRFREQ_'
 'PAMIN11_' 'PAMIN21_' 'PAVIG11_' 'PAVIG21_' '_PACAT1' '_PA150R2'
 '_PA300R2' '_LMTACT1' '_LMTWRK1' '_LMTSCL1' '_RFSEAT2' '_RFSEAT3'
 '_FLSHOT6' '_PNEUMO2' '_AIDTST3']


Before we start handling the NANs, the data should be splitted into numerical data and categorical datd and only pure categorical data needs one_hot. Some categorical data are still treated as numerical data (see `headers_numeric_cat`) because in their cases, larger cateogrical numbers do correspond to larger underlying values they are measuring.

The placement of these variables are done manually by checking the codebook. 

In [ ]:
headers_numeric_cat = ["_AGEG5YR", "_AGE_G", "_BMI5CAT", "_CHLDCNT", "_EDUCAG", "_INCOMG", "_MISVEGN",
                        "ACTIN11_", "ACTIN21_", "_PACAT1", "_PA150R2", "_PA300R2", "_SMOKER3"]
headers_cat = ["_CHOLCHK", "_ASTHMS1", "_PRACE1", "_MRACE1", "_RACE", "_RACEGR3", "_RACE_G1",
                        "_PAREC1", "_LMTACT1", "_LMTWRK1", "_LMTSCL1", ]
headers_numeric = ["_AGE80", "HTIN4", "HTM4", "WTKG3", "_BMI5", "DROCDY3_", "_DRNKWEK", "FTJUDA1_",
                        "FRUTDA1_", "BEANDAY_", "GRENDAY_", "ORNGDAY_", "VEGEDA1_", "_FRUTSUM", "_VEGESUM", 
                        "METVL11_", "METVL21_", "MAXVO2_", "FC60_", "PADUR1_", "PADUR2_", "PAFREQ1_", "PAFREQ2_",
                        "_MINAC11", "_MINAC21", "STRFREQ_", "PAMIN11_", "PAMIN21_", "PA1MIN_", "PAVIG11_", "PAVIG21_",
                        "PA1VIGM_"]

# Every other variable not in this list is a binary variable, i.e. a variable who, apart from missing/unknown flag, 
# only has binary values. 
# We then filter out those that are in the drop list

headers_num_cat_cleaned = [header for header in headers_numeric_cat if header not in drop_list]
headers_cat_cleaned = [header for header in headers_cat if header not in drop_list]
headers_num_cleaned = [header for header in headers_numeric if header not in drop_list]

print("Numerical-categorical (ordinal) headers:", len(headers_num_cat_cleaned))
print(headers_num_cat_cleaned)
print("Categorical headers:", len(headers_cat_cleaned))
print(headers_cat_cleaned)
print("Numerical headers:", len(headers_num_cleaned))
print(headers_num_cleaned)

Numerical-categorical (ordinal) headers: 9
['_CHLDCNT', '_EDUCAG', '_INCOMG', 'ACTIN11_', 'ACTIN21_', '_PACAT1', '_PA150R2', '_PA300R2', '_SMOKER3']
Categorical headers: 6
['_CHOLCHK', '_ASTHMS1', '_RACE', '_LMTACT1', '_LMTWRK1', '_LMTSCL1']
Numerical headers: 22
['_AGE80', 'HTM4', 'WTKG3', '_BMI5', 'DROCDY3_', '_DRNKWEK', '_FRUTSUM', '_VEGESUM', 'METVL11_', 'METVL21_', 'FC60_', 'PADUR1_', 'PADUR2_', 'PAFREQ1_', 'PAFREQ2_', '_MINAC11', '_MINAC21', 'STRFREQ_', 'PAMIN11_', 'PAMIN21_', 'PAVIG11_', 'PAVIG21_']


NAN handling to be done...